# Data Wrangling and Target Creation

**IBM Data Science Capstone — SpaceX Falcon 9**

Repository: [https://github.com/MCerros/IBM-Data-Science-Capstone-SpaceX](https://github.com/MCerros/IBM-Data-Science-Capstone-SpaceX)

Direct notebook URL after upload:  
[https://github.com/MCerros/IBM-Data-Science-Capstone-SpaceX/blob/main/03_SpaceX_Data_Wrangling.ipynb](https://github.com/MCerros/IBM-Data-Science-Capstone-SpaceX/blob/main/03_SpaceX_Data_Wrangling.ipynb)

**Data integrity note:** This notebook uses the project CSV files stored in the same repository.
No rows, metrics, charts, or model scores are manually invented.

## Objective

Inspect missing data and categorical distributions, convert landing outcomes into
the binary target `Class`, and validate the result against `dataset_part_2.csv`.

`Class = 1` represents a successful first-stage recovery.  
`Class = 0` represents an unsuccessful / unrecovered first stage.

In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv("dataset_part_1.csv")
print("Input shape:", data.shape)
data.head()

Input shape: (90, 17)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


In [2]:
missing_pct = (data.isna().mean() * 100).round(2).sort_values(ascending=False)
missing_pct[missing_pct > 0]

LandingPad    28.89
dtype: float64

In [3]:
print("Launches per launch site:")
display(data["LaunchSite"].value_counts().to_frame("Launches"))

print("\nOrbit distribution:")
display(data["Orbit"].value_counts().to_frame("Launches"))

print("\nLanding outcomes:")
display(data["Outcome"].value_counts().to_frame("Launches"))

Launches per launch site:


,Launches
LaunchSite,
CCAFS SLC 40,55
KSC LC 39A,22
VAFB SLC 4E,13



Orbit distribution:


,Launches
Orbit,
GTO,27
ISS,21
VLEO,14
PO,9
LEO,7
SSO,5
MEO,3
HEO,1
ES-L1,1



Landing outcomes:


,Launches
Outcome,
True ASDS,41
None None,19
True RTLS,14
False ASDS,6
True Ocean,5
False Ocean,2
None ASDS,2
False RTLS,1


In [4]:
bad_outcomes = {
    "False ASDS",
    "False Ocean",
    "False RTLS",
    "None ASDS",
    "None None",
}

wrangled = data.copy()
wrangled["Class"] = wrangled["Outcome"].apply(
    lambda outcome: 0 if outcome in bad_outcomes else 1
)

print("Class counts:")
print(wrangled["Class"].value_counts().sort_index())
print(f"Landing success rate: {wrangled['Class'].mean():.2%}")

Class counts:
Class
0    30
1    60
Name: count, dtype: int64
Landing success rate: 66.67%


In [5]:
saved = pd.read_csv("dataset_part_2.csv")

print("Matches saved dataset shape:", wrangled.shape == saved.shape)
print("Matches saved column order:", list(wrangled.columns) == list(saved.columns))
print("Derived Class matches saved Class exactly:", wrangled["Class"].equals(saved["Class"]))
print("Output shape:", saved.shape)
saved.head()

Matches saved dataset shape: True
Matches saved column order: True
Derived Class matches saved Class exactly: True
Output shape: (90, 18)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,0
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,0
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,0
